# BBEH × PromptPotter

Runs PromptPotter's L1/L2/L3 optimization loop on BBEH mini (23 tasks × 10 train / 10 test) using `gpt-oss-120b` via Groq, producing a `results_potter.json` with the same schema as `bbeh_capo.ipynb` and `bbeh_dspy.ipynb`.

**Runs locally against this repo** (unlike the Colab-based CAPO/DSPy notebooks). Prereqs:

- `pip install -e ".[dev,jupyter]"` from the repo root
- `datasets` package: `pip install datasets`
- `.env` with `GROQ_API_KEY` and `LOCAL_SCORING_SECRET`

**Structure** mirrors `notebooks/optimization_campaign.ipynb`: setup → data → per-task optimize → results. No backend server required — uses `LLMOnlyAdapter`. No sensitivity-scan phase (not useful for a head-to-head comparison run).

**Hyperparameters** (`MAX_ROUNDS`, `N_VARIANTS`, `SP_BUDGET_TTEST`) are **unmeasured starting points**, not tuned values — this is a pre-hyperparameter-measurement run.

In [ ]:
# Cell 1 — env + autoreload + path setup
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

# Make shared_config.py importable (this notebook lives next to it).
_HERE = Path.cwd()
if str(_HERE) not in sys.path:
    sys.path.insert(0, str(_HERE))

# Load .env from repo root (two levels up from docs/research/bbeh-comparison/).
try:
    from dotenv import load_dotenv
    load_dotenv(_HERE.parents[2] / ".env")
except ImportError:
    pass

assert os.environ.get("GROQ_API_KEY"), "GROQ_API_KEY missing from environment"
assert os.environ.get("LOCAL_SCORING_SECRET"), "LOCAL_SCORING_SECRET missing from environment"
print("env OK")

In [ ]:
# Cell 2 — PromptPotter notebook API + BBEH data
from promptpotter.presentation.ui.campaign import (
    init_services,
    prepare_scoring_context,
    run_optimization_notebook,
    show_campaign_summary,
    configure_pipeline,
)
from promptpotter.shared.scoring import SCORING_FUNCTIONS

from shared_config import (
    MODEL_ID,
    SPLIT_SEED,
    load_and_split,
    export_results,
)

# Register exact_match into PromptPotter's scoring namespace (once per process).
def _exact_match(predicted, ground_truth):
    p = (predicted or "").strip().lower()
    g = (ground_truth or "").strip().lower()
    return 1.0 if p == g else 0.0

SCORING_FUNCTIONS["exact_match"] = _exact_match

train_by_task, test_by_task = load_and_split()
tasks = sorted(train_by_task.keys())
print(
    f"Loaded {len(tasks)} BBEH tasks, "
    f"{sum(len(v) for v in train_by_task.values())} train, "
    f"{sum(len(v) for v in test_by_task.values())} test (seed={SPLIT_SEED})"
)

In [ ]:
# Cell 3 — campaign config template
#
# These three knobs are the most expensive dials; they are *unmeasured* starting
# points, not tuned values. A later hyperparameter sweep will replace them.
MAX_ROUNDS = 3
N_VARIANTS = 5
SP_BUDGET_TTEST = 10  # == train size per task

TASKS_TO_RUN = tasks  # override to `tasks[:2]` for a smoke test

def build_campaign_config(task: str) -> dict:
    return {
        "dataset_name": "bbeh",
        "dataset_type": "llm-only",
        "scoring": "exact_match(predicted, ground_truth)",
        "sp_budget_ttest": SP_BUDGET_TTEST,
        "recon_sample_size": SP_BUDGET_TTEST,
        "exclude_nodes": [],
        "pipeline_overrides": {},
        "task_context": {
            "task_description": (
                f"Solve the following '{task.replace('_', ' ')}' problem from the BBEH benchmark. "
                "Provide only the final answer, nothing else."
            ),
        },
        "optimization": {
            "l1_patience": 2,
            "max_rounds": MAX_ROUNDS,
            "n_variants": N_VARIANTS,
            "creativity": 0.7,
            "improvement_threshold": 0.01,
            "seed": 42,
            "max_failures": 10,
            "degradation_threshold": 0.4,
            "enable_l2": True,
            "enable_l3": True,
            "l2_patience": 2,
            "l3_patience": 1,
            "l2_temperature": 0.3,
            "l3_temperature": 0.5,
            "enable_critique": True,
        },
        "optimizer_llm": {
            "model": "openai/gpt-oss-120b",
            "provider": "groq",
            "temperature": 0.4,
            "max_tokens": 2000,
        },
        "pipeline_params": None,
    }

def normalize(examples):
    """BBEH {input, target} -> PromptPotter {query, ground_truth}."""
    return [
        {"query": ex["input"], "ground_truth": ex["target"]}
        for ex in examples
    ]

print(f"Will optimize {len(TASKS_TO_RUN)} tasks with max_rounds={MAX_ROUNDS}, "
      f"n_variants={N_VARIANTS}, sp_budget_ttest={SP_BUDGET_TTEST}")

In [ ]:
# Cell 4 — per-task optimization loop
#
# For each BBEH task: fresh session (new cycle_id), run optimize on train split,
# evaluate winner on held-out test split, stash results.

per_task_results: dict[str, dict] = {}
optimized_prompts: dict[str, str] = {}

async def evaluate_winner_on_test(session, winner_pipeline_params, test_norm):
    """Run the winning searchpoint against the held-out test split."""
    hits = 0
    for ex in test_norm:
        resp = await session.backend_client.run_query(
            ex["query"], pipeline_params=winner_pipeline_params
        )
        data = resp.get("data", {})
        ranking = data.get("final_ranking") or []
        predicted = ranking[0].get("candidate", "") if ranking else ""
        hits += int(_exact_match(predicted, ex["ground_truth"]))
    return hits / len(test_norm) if test_norm else 0.0

for i, task in enumerate(TASKS_TO_RUN, start=1):
    print(f"\n{'=' * 60}\n[{i}/{len(TASKS_TO_RUN)}] TASK: {task}\n{'=' * 60}")

    train_norm = normalize(train_by_task[task])
    test_norm = normalize(test_by_task[task])

    session = await init_services(
        dataset_name="bbeh",
        dataset_type="llm-only",
        local_scoring_token=os.environ["LOCAL_SCORING_SECRET"],
    )
    campaign_config = build_campaign_config(task)
    pipeline_params = configure_pipeline(session, campaign_config)

    baseline_ps, dataset_obj, campaign_rounds, _ = await prepare_scoring_context(
        session,
        train_norm,
        campaign_config,
        run_baseline=False,
        pipeline_params=pipeline_params,
    )

    campaign_rounds, _cycle_result = await run_optimization_notebook(
        campaign_rounds,
        dataset_obj,
        campaign_config,
        session=session,
        pipeline_params=pipeline_params,
    )

    # Pick the best round (highest accuracy) as the task winner.
    best_round = max(campaign_rounds, key=lambda r: r.get("accuracy", 0.0))
    winner_prompt_fields = best_round["prompt_fields"]
    winner_pipeline_params = best_round.get("pipeline_params") or pipeline_params
    train_acc = float(best_round.get("accuracy", 0.0))
    baseline_acc = float(campaign_rounds[0].get("accuracy", 0.0))

    test_acc = await evaluate_winner_on_test(
        session, winner_pipeline_params, test_norm
    )

    per_task_results[task] = {
        "accuracy": round(test_acc, 4),
        "n_test": len(test_norm),
        "train_accuracy": round(train_acc, 4),
        "baseline_train_accuracy": round(baseline_acc, 4),
        "rounds": len(campaign_rounds),
    }
    optimized_prompts[task] = winner_prompt_fields.render() if hasattr(
        winner_prompt_fields, "render"
    ) else str(winner_prompt_fields)

    print(
        f"[{i}/{len(TASKS_TO_RUN)}] {task}: "
        f"baseline_train={baseline_acc:.1%} optimized_train={train_acc:.1%} test={test_acc:.1%}"
    )

    await session.backend_client.aclose()

print("\nAll tasks complete.")

In [ ]:
# Cell 5 — export results_potter.json (schema matches bbeh_capo / bbeh_dspy)
export_results(
    method="promptpotter",
    per_task=per_task_results,
    config={
        "optimizer": "promptpotter",
        "max_rounds": MAX_ROUNDS,
        "n_variants": N_VARIANTS,
        "sp_budget_ttest": SP_BUDGET_TTEST,
        "model_id": MODEL_ID,
        "note": "unmeasured starting hyperparameters — pre-sweep",
    },
    optimized_prompts=optimized_prompts,
    output_path="results_potter.json",
)

## Interpretation

`results_potter.json` now sits next to `results_capo.json` and `results_dspy.json` (when those have been run) with an identical top-level schema. The `config.note` field flags that PromptPotter's hyperparameters here are untuned — any head-to-head number below should be read as a floor, not a ceiling, for PromptPotter on BBEH.

Next steps:
- Hyperparameter sweep over `MAX_ROUNDS`, `N_VARIANTS`, `SP_BUDGET_TTEST`.
- Enable sensitivity scan (recon) per task once BBEH-specific `recon_variants.json` is authored.
- Feed the three `results_*.json` files into `docs/research/table-sup-1.md` for the comparison table.